<a href="https://colab.research.google.com/github/braim/nids-tta/blob/main/colab/2-res-ae/NIDS_2_2_0_CCTA_Anom_Residual_AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This experiment pre-trains a Hybrid Residual Autoencoder model on a source network flow dataset to classify and reconstruct data. It then applies Continual Test-Time Adaptation (CTTA) to adapt the model's encoder for improved performance on a target dataset under domain shift.

In [ ]:
# Install necessary libraries and import
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os
import polars as pl
import kagglehub
import gc
from sklearn.preprocessing import StandardScaler
from dataclasses import dataclass


In [ ]:
@dataclass
class Hyperparameters:
    input_dim: int = 43        # Standard for NF- datasets
    num_classes: int = 5       # Benign + 4 Attack Categories
    d_model: int = 128         # Hidden dimension size
    n_blocks: int = 3          # Depth of ResNet
    batch_size: int = 64
    lr_source: float = 1e-3      # Learning rate for pre-training
    lr_adapt: float = 1e-4       # Learning rate for CTTA (usually lower)
    epochs_source: int = 10    # Pre-training epochs
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

config = Hyperparameters()
print(f"Hyperparameters: {config}")

In [ ]:

# ==========================================
# 2. MODEL DEFINITION: Hybrid Residual AE
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.BatchNorm1d(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.BatchNorm1d(d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return x + self.block(x)  # Skip Connection

class HybridResidualAE(nn.Module):
    def __init__(self, input_dim=43, num_classes=5, d_model=128, n_blocks=3):
        super().__init__()

        # --- Shared Encoder ---
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.BatchNorm1d(d_model),
            nn.ReLU()
        )
        self.encoder_blocks = nn.ModuleList([
            ResidualBlock(d_model) for _ in range(n_blocks)
        ])

        # Latent Bottleneck
        self.bottleneck_dim = d_model // 2
        self.to_latent = nn.Linear(d_model, self.bottleneck_dim)

        # --- Head A: Decoder (Reconstruction) ---
        # Used for Unsupervised Adaptation
        self.from_latent = nn.Linear(self.bottleneck_dim, d_model)
        self.decoder_blocks = nn.ModuleList([
            ResidualBlock(d_model) for _ in range(n_blocks)
        ])
        self.output_proj = nn.Linear(d_model, input_dim)

        # --- Head B: Classifier (Prediction) ---
        # Frozen during adaptation
        self.classifier = nn.Sequential(
            nn.Linear(self.bottleneck_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Encoding
        z = self.input_proj(x)
        for block in self.encoder_blocks:
            z = block(z)
        latent = self.to_latent(z)

        # Decoding
        z_rec = self.from_latent(latent)
        for block in self.decoder_blocks:
            z_rec = block(z_rec)
        x_recon = self.output_proj(z_rec)

        # Classification
        logits = self.classifier(latent)

        return logits, x_recon, latent


# ==========================================
# 4. PHASE 1: PRE-TRAINING (SOURCE)
# ==========================================
def train_source(model, train_loader, epochs=10, device='cuda'):
    model.to(device)
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=config.lr_source)

    # Class weights for imbalance (assuming class 0 is majority)
    weights = torch.tensor([0.1, 1.0, 1.0, 1.0, 1.0]).to(device)
    criterion_cls = nn.CrossEntropyLoss(weight=weights)
    criterion_rec = nn.MSELoss()

    print(f"\n--- [Phase 1] Pre-training on Source Domain ({epochs} epochs) ---")

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits, x_recon, _ = model(x)

            # Hybrid Loss: Learn to Classify AND Reconstruct
            loss_cls = criterion_cls(logits, y)
            loss_rec = criterion_rec(x_recon, x)
            loss = loss_cls + 0.5 * loss_rec

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

        avg_loss = total_loss / len(train_loader)
        acc = 100 * correct / total
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

    return model

# ==========================================
# 5. PHASE 2: EVALUATION (STATIC)
# ==========================================
def evaluate_model(model, loader, device='cuda', desc="Dataset"):
    model.eval()
    model.to(device)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _, _ = model(x)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro')

    print(f"[{desc}] Accuracy: {acc:.2%} | Macro F1: {f1:.4f}")
    return acc, f1

# ==========================================
# 6. PHASE 3: CONTINUAL ADAPTATION (CTTA)
# ==========================================
def adapt_stream(model, stream_loader, device='cuda'):
    """
    The Core Novelty: Updates Encoder using Reconstruction Loss on Unlabeled Stream.
    """
    model.to(device)

    # --- A. Freeze Classifier ---
    for param in model.classifier.parameters():
        param.requires_grad = False

    # --- B. Optimizer for Encoder/Decoder only ---
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config.lr_adapt)
    criterion_rec = nn.MSELoss()

    print(f"\n--- [Phase 3] Starting Continual Test-Time Adaptation ---")

    history_acc = []
    history_f1 = []

    # Switch to train mode (but Classifier is frozen via requires_grad=False)
    # We keep BN in train mode to adapt stats (Standard TENT procedure)
    model.train()

    for i, (x_stream, y_true) in enumerate(stream_loader):
        x_stream, y_true = x_stream.to(device), y_true.to(device)

        # -----------------------------
        # 1. Evaluate BEFORE Update (Online Metric)
        # -----------------------------
        with torch.no_grad():
            model.eval() # Temporarily eval for consistent prediction
            logits, _, _ = model(x_stream)
            preds = torch.argmax(logits, dim=1)
            acc = accuracy_score(y_true.cpu(), preds.cpu())
            history_acc.append(acc)
            model.train() # Back to train for update

        # -----------------------------
        # 2. Self-Supervised Update
        # -----------------------------
        optimizer.zero_grad()

        # We ignore labels here! strictly unsupervised.
        _, x_recon, _ = model(x_stream)

        # Loss is pure reconstruction error
        loss = criterion_rec(x_recon, x_stream)

        loss.backward()
        optimizer.step()

        if i % 10 == 0:
            print(f"Batch {i} | Stream Acc: {acc:.2%} | Recon Loss: {loss.item():.4f}")

    avg_acc = np.mean(history_acc)
    print(f"--- Stream Adaptation Complete. Avg Accuracy: {avg_acc:.2%} ---")
    return avg_acc


In [ ]:


def load_dataset(dataset_name):
    """
    Downloads and preprocesses the dataset. Returns X (features) and y (labels).
    """
    print(f"\n[Info] Loading {dataset_name}...")
    try:
        # Download
        download_path = kagglehub.dataset_download(dataset_name)
        csv_files = [os.path.join(r, f) for r, d, files in os.walk(download_path) for f in files if f.endswith('.csv')]
        if not csv_files: raise FileNotFoundError("No CSV found")
        df = pl.read_csv(csv_files[0], infer_schema_length=10000)

        # Preprocess
        if "Attack" in df.columns: df = df.drop("Attack")
        for col in ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']:
            if col in df.columns: df = df.with_columns(pl.col(col).cast(pl.Categorical).to_physical())

        y = df["Label"].to_numpy()
        X = df.drop("Label").to_numpy().astype(np.float32)
        del df; gc.collect()

        # Impute
        if np.isinf(X).any(): X[np.isinf(X)] = np.nan
        if np.isnan(X).any():
            col_mean = np.nanmean(X, axis=0)
            inds = np.where(np.isnan(X))
            X[inds] = np.take(col_mean, inds[1])

        # Scale
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        torch.cuda.empty_cache()
        gc.collect()

        return X, y

    except Exception as e:
        print(f"[Error] Loading failed: {e}")
        return None, None

In [ ]:
# List of datasets to process
datasets = [
    'seyhed/nf-unsw-nb15-v3',
    'seyhed/nf-ton-iot-v3',
    'seyhed/nf-cicids2018-v3'
]

# Execute load_dataset for each
print("Processing datasets...")
for dataset in datasets:
    load_dataset(dataset)

Processing datasets...

[Info] Loading seyhed/nf-unsw-nb15-v3...


100%|██████████| 112M/112M [00:00<00:00, 138MB/s]

Extracting files...



[Info] Loading seyhed/nf-ton-iot-v3...


100%|██████████| 433M/433M [00:06<00:00, 74.2MB/s]

Extracting files...


Following cell is just to check datasets. can be ignored in general

In [ ]:




def _get_feature_names_from_raw_dataset(dataset_name):
    """
    Helper function to get feature names from the raw dataset, mirroring
    preprocessing steps that affect column names in load_dataset.
    """
    try:
        # Download (assuming kagglehub and os are available from previous cell)
        download_path = kagglehub.dataset_download(dataset_name)
        csv_files = [os.path.join(r, f) for r, d, files in os.walk(download_path) for f in files if f.endswith('.csv')]
        if not csv_files: raise FileNotFoundError("No CSV found")
        df = pl.read_csv(csv_files[0], infer_schema_length=10000)

        # Mirror preprocessing from load_dataset for column names
        if "Attack" in df.columns: df = df.drop("Attack")
        # IPV4_SRC_ADDR, IPV4_DST_ADDR casting doesn't change column names, so no need to replicate.

        # The features are all columns EXCEPT 'Label'
        feature_names = [col for col in df.columns if col != "Label"]
        return set(feature_names) # Return as a set for efficient comparison

    except Exception as e:
        print(f"  [Error in _get_feature_names_from_raw_dataset for {dataset_name}] {e}")
        return None


print("\n--- Checking Feature Similarity Across Datasets ---")

datasets_to_check = [
    'seyhed/nf-unsw-nb15-v3',
    'seyhed/nf-ton-iot-v3',
    'seyhed/nf-cicids2018-v3'
]

feature_dims = {}
reference_feature_names_set = None
all_features_match_names = True

for i, dataset_name in enumerate(datasets_to_check):
    print(f"Loading {dataset_name} to check features...")

    # Get feature names for comparison
    current_feature_names_set = _get_feature_names_from_raw_dataset(dataset_name)

    # Get X for dimension check (using the original load_dataset function from another cell)
    X, _ = load_dataset(dataset_name)

    if X is not None and current_feature_names_set is not None:
        feature_dims[dataset_name] = X.shape[1]
        print(f"  {dataset_name}: {X.shape[1]} features (count)")

        if i == 0:
            reference_feature_names_set = current_feature_names_set
            print(f"  Reference features established from {dataset_name}.")
        else:
            # Check for features present in current but not in reference
            new_features = current_feature_names_set - reference_feature_names_set
            if new_features:
                all_features_match_names = False
                print(f"  [Warning] {dataset_name} has new features not in reference: {sorted(list(new_features))}")

            # Check for features present in reference but not in current
            missing_features = reference_feature_names_set - current_feature_names_set
            if missing_features:
                all_features_match_names = False
                print(f"  [Warning] {dataset_name} is missing features present in reference: {sorted(list(missing_features))}")

            if not new_features and not missing_features:
                print(f"  {dataset_name} has the same set of feature names as the reference dataset.")

    else:
        print(f"  Failed to load {dataset_name} for feature check. Skipping name comparison.")
        all_features_match_names = False # If loading fails, cannot confirm name match

print("\n--- Summary of Feature Dimensions (Count) ---")
for dataset, dim in feature_dims.items():
    print(f"- {dataset}: {dim} features")

# Check if all dimensions (counts) are the same
if len(set(feature_dims.values())) == 1 and len(feature_dims) == len(datasets_to_check):
    print("\nAll loaded datasets have the same number of features (count).")
elif len(feature_dims) == len(datasets_to_check):
    print("\nDatasets have different numbers of features (count). Consider alignment if needed for combined models.")
else:
    print("\nCould not load all datasets or determine feature similarity completely.")

print("\n--- Summary of Feature Names ---")
if all_features_match_names:
    print("All datasets have the same set of feature names as the first dataset.")
else:
    print("Feature names differ across datasets. Review the warnings above for details.")


--- Checking Feature Similarity Across Datasets ---
Loading seyhed/nf-unsw-nb15-v3 to check features...
Using Colab cache for faster access to the 'nf-unsw-nb15-v3' dataset.

[Info] Loading seyhed/nf-unsw-nb15-v3...
Using Colab cache for faster access to the 'nf-unsw-nb15-v3' dataset.
  seyhed/nf-unsw-nb15-v3: 53 features (count)
  Reference features established from seyhed/nf-unsw-nb15-v3.
Loading seyhed/nf-ton-iot-v3 to check features...
Using Colab cache for faster access to the 'nf-ton-iot-v3' dataset.

[Info] Loading seyhed/nf-ton-iot-v3...
Using Colab cache for faster access to the 'nf-ton-iot-v3' dataset.
  seyhed/nf-ton-iot-v3: 53 features (count)
  seyhed/nf-ton-iot-v3 has the same set of feature names as the reference dataset.
Loading seyhed/nf-cicids2018-v3 to check features...
Using Colab cache for faster access to the 'nf-cicids2018-v3' dataset.

[Info] Loading seyhed/nf-cicids2018-v3...
Using Colab cache for faster access to the 'nf-cicids2018-v3' dataset.
  seyhed/nf-ci

In [ ]:
# ==========================================
# 7. MAIN EXECUTION
# ==========================================
# NOTE: Executing this cell requires the previous cell (Definitions) to be run first.

if __name__ == "__main__":
    print(f"Using Device: {config.device}")

    # --- Step 1: Data Setup ---
    # 1. Load Source
    print("Loading Source Data: seyhed/nf-unsw-nb15-v3...")
    X_src, y_src = load_dataset('seyhed/nf-unsw-nb15-v3')

    if X_src is None:
        raise RuntimeError("Failed to load source dataset!")

    # Update dimensions based on loaded data
    config.input_dim = X_src.shape[1]
    # Note: We keep config.num_classes as 5 to match the pre-defined model/training weights,
    # even if the loaded dataset is binary (classes 0 and 1 are valid indices).

    print(f"Source Loaded. Samples: {len(y_src)}, Features: {config.input_dim}")

    # Create Source Dataset
    ds_source = TensorDataset(torch.tensor(X_src), torch.tensor(y_src, dtype=torch.long))

    # 2. Load Target
    print("Loading Target Data: seyhed/nf-ton-iot-v3...")
    X_tgt, y_tgt = load_dataset('seyhed/nf-ton-iot-v3')

    if X_tgt is None:
        raise RuntimeError("Failed to load target dataset!")

    print(f"Target Loaded. Samples: {len(y_tgt)}, Features: {X_tgt.shape[1]}")

    # Basic Feature Alignment (Just in case dimensions differ)
    if X_tgt.shape[1] != config.input_dim:
        print(f"[Warning] Feature mismatch: Source {config.input_dim} vs Target {X_tgt.shape[1]}. Adjusting Target...")
        if X_tgt.shape[1] > config.input_dim:
            X_tgt = X_tgt[:, :config.input_dim]
        else:
            padding = np.zeros((X_tgt.shape[0], config.input_dim - X_tgt.shape[1]), dtype=X_tgt.dtype)
            X_tgt = np.hstack([X_tgt, padding])

    # Create Target Dataset
    ds_target = TensorDataset(torch.tensor(X_tgt), torch.tensor(y_tgt, dtype=torch.long))

    loader_source = DataLoader(ds_source, batch_size=config.batch_size, shuffle=True)
    loader_target_eval = DataLoader(ds_target, batch_size=config.batch_size, shuffle=False)
    loader_target_stream = DataLoader(ds_target, batch_size=config.batch_size, shuffle=False)

    # --- Step 2: Initialize Model ---
    model = HybridResidualAE(
        input_dim=config.input_dim,
        num_classes=config.num_classes,
        d_model=config.d_model,
        n_blocks=config.n_blocks
    )

    # --- Step 3: Train Source ---
    model = train_source(model, loader_source, epochs=config.epochs_source, device=config.device)

    # --- Step 4: Baseline Evaluation ---
    print("\n" + "="*40)
    print("BASELINE EVALUATION (NO ADAPTATION)")
    print("="*40)
    acc_src, _ = evaluate_model(model, loader_source, device=config.device, desc="Source Test")
    acc_tgt_before, _ = evaluate_model(model, loader_target_eval, device=config.device, desc="Target Test (Before Adapt)")

    # --- Step 5: Run CTTA ---
    print("\n" + "="*40)
    print("RUNNING ADAPTATION (CTTA)")
    print("="*40)
    # NOTE: We iterate through the target stream ONE BATCH AT A TIME
    acc_tgt_after = adapt_stream(model, loader_target_stream, device=config.device)

    # --- Step 6: Final Report ---
    print("\n" + "="*40)
    print("FINAL RESULTS")
    print("="*40)
    print(f"Source Accuracy:              {acc_src:.2%}")
    print(f"Target Accuracy (Static):     {acc_tgt_before:.2%}")
    print(f"Target Accuracy (Adaptive):   {acc_tgt_after:.2%}")

    improvement = acc_tgt_after - acc_tgt_before
    print(f"Improvement:                  {improvement:+.2%} points")


Using Device: cpu
Loading Source Data: seyhed/nf-unsw-nb15-v3...

[Info] Loading seyhed/nf-unsw-nb15-v3...


100%|██████████| 112M/112M [00:00<00:00, 161MB/s]

Extracting files...


Streaming output truncated to the last 5000 lines.
Batch 70450 | Stream Acc: 42.19% | Recon Loss: 0.0084
Batch 70460 | Stream Acc: 67.19% | Recon Loss: 0.0047
Batch 70470 | Stream Acc: 87.50% | Recon Loss: 0.0056
Batch 70480 | Stream Acc: 35.94% | Recon Loss: 0.0051
Batch 70490 | Stream Acc: 65.62% | Recon Loss: 0.0054
Batch 70500 | Stream Acc: 71.88% | Recon Loss: 0.0061
Batch 70510 | Stream Acc: 51.56% | Recon Loss: 0.0055
Batch 70520 | Stream Acc: 23.44% | Recon Loss: 0.0063
Batch 70530 | Stream Acc: 7.81% | Recon Loss: 0.0127
Batch 70540 | Stream Acc: 26.56% | Recon Loss: 0.0101
Batch 70550 | Stream Acc: 65.62% | Recon Loss: 0.0068
Batch 70560 | Stream Acc: 40.62% | Recon Loss: 0.0035
Batch 70570 | Stream Acc: 42.19% | Recon Loss: 0.0044
Batch 70580 | Stream Acc: 40.62% | Recon Loss: 0.0028
Batch 70590 | Stream Acc: 10.94% | Recon Loss: 0.0171
Batch 70600 | Stream Acc: 57.81% | Recon Loss: 0.0048
Batch 70610 | Stream Acc: 48.44% | Recon Loss: 0.0071
Batch 70620 | Stream Acc: 53.12%